In [5]:
import pandas as pd
import plotly.graph_objects as go
#========================
# Load data
# ================================================================
RUN_LABEL = "s2_balanced"
SCALER    = "minmax"
BASE_DIR  = f"Results/Regular_clustering/Full_dataset/With_counts/{SCALER}/{RUN_LABEL}"
mcs = 2271
ms = 15
df = pd.read_csv(f"{BASE_DIR}/final_mcs{mcs}_ms{ms}/clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv", low_memory=False)
output_dir = (f"{BASE_DIR}/final_mcs{mcs}_ms{ms}")

# ================================================================
# SHARED CONFIGURATION
# ================================================================
COL_TRANSPORT = 'transport_grouped'
COL_TRI       = 'triage'
COL_CLUSTER   = 'cluster_label'          # column with labels like "C1 - UHCD..."
                                          # adjust if your column is named differently

MEASURE_COLS = [
    'is_temp_measured', 'is_sat_measured', 'is_hr_measured',
    'is_bp_measured', 'is_rr_measured', 'is_pupils_measured',
    'is_o2_measured', 'is_hemocue_measured', 'is_breathalyzer_measured',
    'is_urine_dipstick_clean_measured', 'is_gcs_measured', 'is_pain_measured',
    'is_cap_blood_sugar_mmol_L_measured'
]

# ================================================================
# COLOR PALETTES
# ================================================================
transport_colors = {
    'Unknown':             '#5a5e6b',
    'Personal':            '#3498db',
    'Post medical advice': '#9b59b6',
    'Ambulance':           '#e67e22',
    'Emergency services':  '#2ecc71',
}

measure_colors = {
    '8+ measures':  '#27ae60',
    '4-7 measures': '#f1c40f',
    '1-3 measures': '#e67e22',
    '0 measures':   '#e74c3c',
}

triage_colors = {
    'Triage 1': '#e74c3c',
    'Triage 2': '#e67e22',
    'Triage 3': '#f1c40f',
    'Triage 4': '#27ae60',
    'Triage 5': '#2980b9',
}

# Cluster colors — ordered C1..C9 + Outliers
cluster_colors = {
    'C1': '#7b241c',
    'C2': '#922b21',
    'C3': '#c0392b',
    'C4': '#e67e22',
    'C5': '#f39c12',
    'C6': '#27ae60',
    'C7': '#2980b9',
    'C8': '#1abc9c',
    'C9': '#95a5a6',
    'Outliers': '#5d6d7e',
}

def get_cluster_color(label):
    """Match cluster label starting with C1..C9 or Outliers."""
    for key, col in cluster_colors.items():
        if str(label).startswith(key):
            return col
    return '#a6acaf'

def get_node_color(label):
    for palette in [transport_colors, measure_colors, triage_colors]:
        if label in palette:
            return palette[label]
    return get_cluster_color(label)

# ================================================================
# DATA PREPARATION  (run once, shared by both figures)
# ================================================================
df_alluvial = df.copy()

df_alluvial['_transport_label'] = df_alluvial[COL_TRANSPORT].fillna('Unknown')
df_alluvial['_tri_label']       = 'Triage ' + df_alluvial[COL_TRI].astype(int).astype(str)

def score_to_cat(x):
    if x == 0:   return '0 measures'
    elif x <= 3: return '1-3 measures'
    elif x <= 7: return '4-7 measures'
    else:        return '8+ measures'

df_alluvial['_measures_score'] = (
    df_alluvial[MEASURE_COLS]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0).sum(axis=1).astype(int)
)
df_alluvial['_measures_label'] = df_alluvial['_measures_score'].apply(score_to_cat)
df_alluvial['_cluster_label']  = df_alluvial[COL_CLUSTER].astype(str)

# ================================================================
# HELPERS
# ================================================================
def get_present_vals(order_list, series):
    present = set(series.dropna().unique())
    return [v for v in order_list if v in present]

def assign_y(labels):
    n = len(labels)
    if n <= 1: return [0.5]
    return [round(0.05 + i * (0.90 / (n - 1)), 4) for i in range(n)]

def make_links(df_sub, col_from, col_to, label_idx):
    # Si col_to est la colonne triage elle-même, on n'ajoute pas _tri_label au groupby
    group_cols = [col_from, col_to]
    if '_tri_label' not in [col_from, col_to]:
        group_cols.append('_tri_label')
        has_triage_color = True
    else:
        has_triage_color = False

    grp = df_sub.groupby(group_cols).size().reset_index(name='count')
    sources, targets, values, colors = [], [], [], []
    for _, row in grp.iterrows():
        src, tgt = str(row[col_from]), str(row[col_to])
        if src in label_idx and tgt in label_idx:
            sources.append(label_idx[src])
            targets.append(label_idx[tgt])
            values.append(int(row['count']))
            if has_triage_color:
                hex_c = triage_colors.get(row['_tri_label'], '#a6acaf')
            else:
                # colorer par la source ou la cible selon le sens
                hex_c = triage_colors.get(str(row[col_to]), triage_colors.get(str(row[col_from]), '#a6acaf'))
            r, g, b = int(hex_c[1:3], 16), int(hex_c[3:5], 16), int(hex_c[5:7], 16)
            colors.append(f'rgba({r},{g},{b},0.35)')
    return sources, targets, values, colors
def build_sankey(col_order, df_alluvial, title, output_name):
    """
    col_order: list of (key_name, label_column, ordered_values_list)
    """
    all_labels = []
    col_vals   = {}
    for key, col, order in col_order:
        vals = get_present_vals(order, df_alluvial[col])
        col_vals[key] = (col, vals)
        all_labels += vals

    label_idx   = {label: i for i, label in enumerate(all_labels)}
    node_colors = [get_node_color(l) for l in all_labels]

    # X positions evenly spaced
    n_cols  = len(col_order)
    x_steps = [round(0.01 + i * (0.98 / (n_cols - 1)), 4) for i in range(n_cols)]
    col_x   = {col_order[i][0]: x_steps[i] for i in range(n_cols)}

    pos = {}
    for key, (col, vals) in col_vals.items():
        for label, y in zip(vals, assign_y(vals)):
            pos[label] = (col_x[key], y)

    node_x = [pos[l][0] for l in all_labels]
    node_y = [pos[l][1] for l in all_labels]

    # Build links between consecutive columns
    s_all, t_all, v_all, c_all = [], [], [], []
    keys = [k for k, _, _ in col_order]
    for i in range(len(keys) - 1):
        col_from = col_vals[keys[i]][0]
        col_to   = col_vals[keys[i+1]][0]
        s, t, v, c = make_links(df_alluvial, col_from, col_to, label_idx)
        s_all += s; t_all += t; v_all += v; c_all += c

    fig = go.Figure(data=[go.Sankey(
        arrangement='fixed',
        node=dict(
            pad=20, thickness=18,
            label=all_labels,
            color=node_colors,
            x=node_x, y=node_y,
            line=dict(color='#cccccc', width=0.5)
        ),
        link=dict(source=s_all, target=t_all, value=v_all, color=c_all)
    )])

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color='black'), x=0.01, y=0.97),
        font=dict(size=11, family='Arial', color='black'),
        paper_bgcolor='white', plot_bgcolor='white',
        template='plotly_white',
        height=950,
        margin=dict(t=150, b=150, l=100, r=150),
    )

    fig.write_html(f"{output_dir}/{output_name}.html")
    fig.write_image(f"{output_dir}/{output_name}.png", width=1400, height=700, scale=2)
    fig.show()
    print(f"Saved: {output_name}")


# ================================================================
# CLUSTER ORDER (shared)
# ================================================================
cluster_order_list = [
    'C1 — UHCD + hospitalization + mixed workup ++',
    'C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+',
    'C3 — Hospitalized + blood test++ CTScan++ ECG+',
    'C4 — Hospitalized + blood test++ MRI+++ ECG++',
    'C5 — Hospitalized + blood test + ECG +/- X-ray ++',
    'C6 — Hospitalized + isolated imaging (xray,ctscan)',
    'C7 — Discharged + biology + ECG+',
    'C8 — Discharged + isolated X-ray',
    'C9 — Discharged + minimal consumption',
    'Outliers',
]



# ================================================================
# VERSION 1 — Transport → Measures → Cluster
# ================================================================
col_order_v1 = [
    ('transport', '_transport_label', ['Emergency services', 'Ambulance', 'Post medical advice', 'Personal', 'Unknown']),
    ('measures',  '_measures_label',  ['8+ measures', '4-7 measures', '1-3 measures', '0 measures']),
    ('cluster',   '_cluster_label',   cluster_order_list),
]

build_sankey(
    col_order_v1,
    df_alluvial,
    title='Patient flow: Transport → Vital sign measurement intensity → Cluster',
    output_name='sankey_transport_measures_cluster'
)


# ================================================================
# VERSION 2 — Transport → Measures → Triage → Cluster
# ================================================================
col_order_v2 = [
    ('transport', '_transport_label', ['Emergency services', 'Ambulance', 'Post medical advice', 'Personal', 'Unknown']),
    ('measures',  '_measures_label',  ['8+ measures', '4-7 measures', '1-3 measures', '0 measures']),
    ('triage',    '_tri_label',       ['Triage 1', 'Triage 2', 'Triage 3', 'Triage 4', 'Triage 5']),
    ('cluster',   '_cluster_label',   cluster_order_list),
]

build_sankey(
    col_order_v2,
    df_alluvial,
    title='Patient flow: Transport → Vital sign measurement intensity → Triage → Cluster',
    output_name='sankey_transport_measures_triage_cluster'
)

/tmp/ipykernel_1482220/2522822314.py:200: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




Saved: sankey_transport_measures_cluster


/tmp/ipykernel_1482220/2522822314.py:200: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




Saved: sankey_transport_measures_triage_cluster


In [4]:
print(df_alluvial['_cluster_label'].value_counts(dropna=False).to_string())


_cluster_label
C9 — Discharged + minimal consumption                                                         12509
C1 — UHCD + hospitalization + mixed workup ++                                                 10306
C8 — Discharged + isolated X-ray                                                               5995
C5 — Hospitalized + blood test + ECG +/- X-ray ++                                              5422
C3 — Hospitalized + blood test++ CTScan++ ECG+                                                 5379
Outliers                                                                                       4695
C7 — Discharged + biology + ECG+                                                               4286
C6 — Hospitalized + isolated imaging (xray,ctscan)                                             2909
C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+     2842
C4 — Hospitalized + blood test++ MRI+++ ECG++                                        